# Análise exploratória — Classificação de porte empresarial

**Etapa 2.** Pré-requisito: rodar `python -m src.pipeline --etapa 2` (gera `data/interim/estabelecimentos.parquet` e `outputs/reports/relatorio_qualidade.md`).

A lógica reutilizável fica em `src/`; este notebook só explora e visualiza.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # raiz do projeto

import pandas as pd
import matplotlib.pyplot as plt
from src import config
from src.data.load_data import load_interim, normalize_text

df = load_interim()
print(df.shape)
df.head()

## Relatório de qualidade (resumo)

In [ ]:
rep = json.loads(config.REPORT_QUALIDADE_JSON.read_text(encoding='utf-8'))
for p in rep['pendencias']:
    print('-', p)
rep['chaves']

## Rótulos por estabelecimento (`Obs Porte Sa3`)

In [ ]:
df['porte_obs'].map(normalize_text).value_counts(dropna=False)

## Capital social

⚠ Semântica do zero **pendente**. O gráfico separa zeros dos valores positivos (escala log).

In [ ]:
cap = df['capital_social']
print('nulos:', cap.isna().sum(), '| zeros:', (cap == 0).sum(), '| positivos:', (cap > 0).sum())
import numpy as np
plt.figure(figsize=(7, 3.5))
plt.hist(np.log10(cap[cap > 0]), bins=50)
plt.xlabel('log10(capital social) — apenas valores > 0')
plt.ylabel('estabelecimentos')
plt.tight_layout(); plt.show()

### Capital social = 0 por grupo de rótulo (evidência para decidir a regra)

In [ ]:
pd.DataFrame(rep['capital_social']['evidencias_semantica_zero']['por_grupo_de_rotulo']).T

## Quantidade de funcionários (cobertura)

In [ ]:
rep['funcionarios']

## Estabelecimentos por CNPJ raiz

In [ ]:
df.groupby('cnpj_raiz').size().value_counts().sort_index().head(15)